# Notebook 01 — Getting Started with Sentinel-2 Remote Sensing

**Study area:** Anantapur and surrounding area, Andhra Pradesh, India  
**Dataset:** Sentinel-2 MSI, Level-2A (surface reflectance)  
**Selected acquisition:** 9 October 2026 (product sensing time shown in Copernicus Browser: `2026-10-09T05:07:19.024000Z`)  
**Tile shown in the downloaded product name:** `T43PGS`  
**Main goal:** Understand the satellite product, inspect the bands, view the image, and calculate a first NDVI map.

> **Important:** This notebook is a learning record and starter workflow. We have identified the files in the downloaded SAFE product, but we have not yet verified the imagery by loading it in QGIS or Python. Run the cells on your computer after installing the required packages and setting the correct folder path.

## 1. What we have done so far

1. Opened the Copernicus Browser and searched for Sentinel-2 imagery around Anantapur.
2. Selected the **Sentinel-2 L2A** collection.
3. Compared the available scenes for **9 October 2026** and **4 October 2026**. The 9 October scene looked less cloudy over the area of interest, so we chose it for the first attempt.
4. Downloaded the Sentinel-2 product as a ZIP file and extracted it.
5. Opened the extracted `.SAFE` product and navigated to a granule's `IMG_DATA` directory.
6. Found the resolution folders: `R10m`, `R20m`, and `R60m`.
7. Inspected `R10m`, which contains files such as `B02`, `B03`, `B04`, `B08`, `TCI`, `AOT`, and `WVP`.

### What has *not* been done yet

- We have not loaded the downloaded raster into QGIS.
- We have not checked the scene's cloud/shadow contamination with a formal quality mask.
- We have not calculated NDVI or interpreted its values from the downloaded pixels.
- We have not compared multiple dates in a controlled change-detection analysis.

A green-looking NDVI layer in the Copernicus Browser is a **visualisation**, not proof that the area is all healthy vegetation. The browser's colour ramp and the actual satellite acquisition both affect the appearance.

## 2. What is Sentinel-2?

Sentinel-2 is an Earth-observation mission carrying the **MultiSpectral Instrument (MSI)**. It records reflected energy in multiple spectral bands. Different materials—vegetation, soil, water, buildings and clouds—reflect or absorb energy differently at different wavelengths.

Sentinel-2 Level-2A products provide atmospherically corrected surface-reflectance data, along with quality/auxiliary information. The bands do not all have the same spatial resolution: commonly used bands are supplied at **10 m, 20 m, or 60 m**.

- **Spatial resolution** tells you the approximate ground size represented by one pixel.
- **Spectral resolution** refers to the wavelength regions measured by the instrument.
- **Temporal resolution** refers to how often the satellite can revisit an area; actual usable observations depend on coverage, clouds and acquisition availability.

The `R10m`, `R20m`, and `R60m` folders group imagery by pixel spacing. They are not different satellite missions.

## 3. Wavelengths and bands

Wavelengths below are **approximate central wavelengths** in nanometres (nm). Exact spectral response differs slightly between Sentinel-2 satellite units, so use these as learning values rather than laboratory-grade specifications.

| Band | Approx. wavelength | Region / common name | Native pixel size | Common uses |
|---|---:|---|---:|---|
| B01 | 443 nm | Coastal aerosol | 60 m | Aerosol/coastal studies |
| B02 | 490 nm | Blue | 10 m | True-colour display; water/coastal studies |
| B03 | 560 nm | Green | 10 m | True-colour display; vegetation/water features |
| B04 | 665 nm | Red | 10 m | True-colour display; chlorophyll absorption; NDVI |
| B05 | 705 nm | Red-edge 1 | 20 m | Vegetation condition/chlorophyll-related analysis |
| B06 | 740 nm | Red-edge 2 | 20 m | Vegetation condition |
| B07 | 783 nm | Red-edge 3 | 20 m | Vegetation structure/condition |
| B08 | 842 nm | Near-infrared (NIR) | 10 m | Vegetation mapping; NDVI |
| B8A | 865 nm | Narrow NIR | 20 m | Vegetation analysis |
| B09 | 945 nm | Water vapour | 60 m | Atmospheric/water-vapour applications |
| B10 | 1375 nm | Cirrus | 60 m | High-cloud detection; not a standard surface-reflectance band for ordinary land analysis |
| B11 | 1610 nm | SWIR 1 | 20 m | Moisture, burn scars, land-cover discrimination |
| B12 | 2190 nm | SWIR 2 | 20 m | Burn scars, moisture and land-cover discrimination |

### Which bands are we using first?

- **TCI_10m.jp2** — a ready-made true-colour image, useful for an initial visual check.
- **B04_10m.jp2 (red)** — the red band used in the NDVI denominator.
- **B08_10m.jp2 (near-infrared)** — the NIR band used in NDVI.

We are starting with 10 m bands because they share the same nominal pixel size. This avoids a resolution mismatch in our first NDVI calculation.

## 4. How to interpret the main bands

### Visible bands: B02, B03, B04

These broadly correspond to the blue, green and red portions of the spectrum. Combining them as **R = B04, G = B03, B = B02** produces a natural-colour image similar to what a person might see from above, although it will not always look exactly like a photograph.

### Near-infrared: B08

Healthy green leaves typically reflect strongly in near-infrared while absorbing much of the red light used for photosynthesis. That contrast is the reason red and NIR are useful for vegetation indices.

### Short-wave infrared: B11 and B12

SWIR bands respond to moisture and other material properties. They are useful in applications such as burn-scar mapping, vegetation/water-content studies and land-cover discrimination. We will explore them after the first multispectral exercise.

### Do not interpret one band in isolation

A bright or dark pixel is not automatically a particular land-cover type. Interpretation depends on the band, display stretch, atmosphere, sun angle, season, soil background, moisture and cloud/shadow conditions.

## 5. NDVI: the first index

The Normalized Difference Vegetation Index (NDVI) is calculated as:

\[
\mathrm{NDVI} = \frac{\mathrm{NIR} - \mathrm{Red}}{\mathrm{NIR} + \mathrm{Red}}
= \frac{B08 - B04}{B08 + B04}
\]

For valid inputs, NDVI is usually between **−1 and +1**.

| Typical pattern | Possible interpretation | Caution |
|---|---|---|
| Clearly negative | Often water, cloud/shadow effects, or other surfaces | Not every negative pixel is water |
| Around zero / low positive | Often bare soil, built-up areas, sparse vegetation, or mixed pixels | These classes overlap |
| Moderate positive | Could be grass, crops, or mixed vegetation | Depends on crop stage and season |
| Higher positive | Often dense, actively green vegetation | High NDVI does not directly measure crop yield or prove plant health |

These are **guidelines, not universal thresholds**. Use local context and, where possible, field data or a land-cover reference to validate an interpretation.

### Reflectance scaling note

Sentinel-2 L2A JP2 bands are often stored as scaled integer values. NDVI is a ratio, so a common multiplicative scale factor cancels if it is the same for both bands. However, **NoData values, masks, and zero denominators must still be handled**. We will inspect metadata and valid ranges before treating results as final.

## 6. Prepare the Python environment

Run this once in a terminal or in a notebook cell if your environment permits package installation:

```bash
python -m pip install rasterio numpy matplotlib
```

If installation fails on your computer, do not repeatedly install packages blindly—share the error message and we can troubleshoot it.

**Folder path:** In File Explorer, navigate to the extracted `.SAFE` product, then open:

`GRANULE → [granule folder] → IMG_DATA → R10m`

Inside that directory, you should find files whose names end with `B04_10m.jp2`, `B08_10m.jp2`, and `TCI_10m.jp2`. Your exact parent-folder names may differ slightly.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# EDIT THIS PATH to the actual R10m folder on your computer.
# Example only — do not copy this path unless it matches your folder:
R10M_DIR = Path(r"D:\\YOUR_FOLDER\\PRODUCT.SAFE\\GRANULE\\YOUR_GRANULE\\IMG_DATA\\R10m")

# List the files so we can confirm the path is correct.
if not R10M_DIR.exists():
    print("Folder not found. Edit R10M_DIR above to point to your extracted R10m folder.")
else:
    for p in sorted(R10M_DIR.glob("*.jp2")):
        print(p.name)

## 7. Load the true-colour image

The `TCI_10m.jp2` file is a pre-composed true-colour image. We use it first to check that the downloaded scene opens and covers the expected area.

The code below uses rasterio to read the image. Depending on the product and software version, the file may be stored with three colour bands. The display stretch is only for viewing; it does not change the source file.

In [ ]:
tci_path = R10M_DIR / next(iter(R10M_DIR.glob("*TCI_10m.jp2")), Path("__missing__"))

if not tci_path.exists():
    print("TCI file not found. Check the R10m folder and filenames.")
else:
    with rasterio.open(tci_path) as src:
        print("Image size (columns, rows):", src.width, src.height)
        print("Number of bands:", src.count)
        print("CRS:", src.crs)
        print("Pixel size:", src.res)
        tci = src.read()

    if tci.shape[0] >= 3:
        rgb = np.moveaxis(tci[:3], 0, -1).astype(np.float32)
        # Percentile stretch for visualisation only
        lo, hi = np.nanpercentile(rgb, [2, 98])
        rgb_display = np.clip((rgb - lo) / (hi - lo + 1e-9), 0, 1)
        plt.figure(figsize=(10, 8))
        plt.imshow(rgb_display)
        plt.title("Sentinel-2 L2A — True Colour Image (display stretch)")
        plt.axis("off")
        plt.show()
    else:
        print("This TCI file did not expose three bands as expected; inspect src.count and metadata.")

## 8. Read the red and NIR bands and calculate NDVI

This cell reads B04 and B08, checks that their grids match, masks NoData pixels, and calculates NDVI. It does **not** classify land cover automatically.

If the grids do not match, stop rather than resampling silently. These two files should normally be compatible because both are 10 m bands from the same granule.

In [ ]:
def read_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
    valid = np.isfinite(arr)
    if nodata is not None:
        valid &= arr != nodata
    return arr, valid, profile, transform, crs

red_path = R10M_DIR / next(iter(R10M_DIR.glob("*B04_10m.jp2")), Path("__missing__"))
nir_path = R10M_DIR / next(iter(R10M_DIR.glob("*B08_10m.jp2")), Path("__missing__"))

if not red_path.exists() or not nir_path.exists():
    print("Could not find both B04_10m.jp2 and B08_10m.jp2. Check R10M_DIR.")
else:
    red, red_valid, red_profile, red_transform, red_crs = read_band(red_path)
    nir, nir_valid, nir_profile, nir_transform, nir_crs = read_band(nir_path)

    same_grid = (
        red.shape == nir.shape
        and red_transform == nir_transform
        and red_crs == nir_crs
    )
    print("Red shape:", red.shape, "| NIR shape:", nir.shape)
    print("Same grid:", same_grid)
    print("Red CRS:", red_crs)

    if not same_grid:
        raise ValueError("B04 and B08 grids differ. Stop and inspect metadata before calculating NDVI.")

    denominator = nir + red
    valid = red_valid & nir_valid & np.isfinite(denominator) & (denominator != 0)

    ndvi = np.full(red.shape, np.nan, dtype=np.float32)
    ndvi[valid] = (nir[valid] - red[valid]) / denominator[valid]

    # Print summary statistics for valid pixels
    print("Valid pixel count:", int(np.isfinite(ndvi).sum()))
    print("NDVI minimum:", float(np.nanmin(ndvi)))
    print("NDVI median:", float(np.nanmedian(ndvi)))
    print("NDVI maximum:", float(np.nanmax(ndvi)))

    plt.figure(figsize=(10, 8))
    plt.imshow(ndvi, vmin=-1, vmax=1, cmap="RdYlGn")
    plt.colorbar(label="NDVI")
    plt.title("First NDVI map — Sentinel-2 B08 and B04")
    plt.axis("off")
    plt.show()

## 9. How to read your NDVI map

1. Look at the **spatial pattern** first: do green areas broadly correspond to fields or vegetated patches visible in the true-colour image?
2. Look for **negative or very low values** around water, bare ground, built-up surfaces, clouds or shadows—but do not assign a class from NDVI alone.
3. Compare the map with the true-colour image. A single index can confuse different surface types.
4. Check for clouds and shadows. The scene looked less cloudy in the browser on 9 October than on 4 October, but clouds can still remain in the selected scene.
5. Record the date and dataset whenever you save a map. Do not compare dates unless the area, processing, resolution and cloud/shadow treatment are consistent.

### A useful scientific habit

Write down what you **observed**, what you **infer**, and what you **still need to verify** separately. For example:

- **Observed:** a patch has a high NDVI value.
- **Possible interpretation:** it may contain dense green vegetation.
- **Still to verify:** crop type, irrigation status, health, or exact land-cover class.

## 10. Save NDVI as a GeoTIFF

Once the NDVI map has been calculated successfully, this cell writes it to a GeoTIFF in the same R10m folder. This is a derived output; it does not overwrite the original Sentinel-2 bands.

```python
# Run only after the NDVI calculation cell has completed successfully.
out_path = R10M_DIR / "Anantapur_NDVI_2026-10-09.tif"

out_profile = red_profile.copy()
out_profile.update(
    driver="GTiff",
    dtype="float32",
    count=1,
    nodata=-9999.0,
    compress="deflate"
)

ndvi_to_save = np.where(np.isfinite(ndvi), ndvi, -9999.0).astype(np.float32)

with rasterio.open(out_path, "w", **out_profile) as dst:
    dst.write(ndvi_to_save, 1)

print("Saved:", out_path)
```

You can then open the GeoTIFF in QGIS and style it with a suitable single-band pseudocolour renderer. The colours used in this notebook are for visual interpretation; they are not a land-cover classification.

## 11. First-notebook checklist

- [ ] Confirm the extracted SAFE product and acquisition date.
- [ ] Locate `TCI_10m.jp2`, `B04_10m.jp2`, and `B08_10m.jp2`.
- [ ] Set `R10M_DIR` to the correct folder.
- [ ] Display the true-colour image.
- [ ] Confirm that B04 and B08 use the same grid.
- [ ] Calculate NDVI and inspect its minimum, median and maximum.
- [ ] Compare the NDVI pattern against the true-colour image.
- [ ] Save the NDVI GeoTIFF and open it in QGIS.

### Notebook 01 outcome

By the end, you should understand the downloaded product structure, the meaning of common Sentinel-2 wavelength bands, and the basic logic of NDVI. The next notebook can focus on **QGIS visualisation, cloud/shadow masking, and comparing vegetation patterns across dates**.

**Data-quality note:** This is a learning workflow, not a validated land-cover or crop-health product. Check the scene metadata and quality masks before making quantitative claims.